# **Licenciatura em Ciências da Computação**

### Aprendizagem Computacional 25/26

# IMDB Sentiment Analysis: Feature Engineering Challenge

## 1. Setup & Data Loading

First, we'll grab the dataset. We'll use a subset (10,000 rows) to keep the processing fast during class.

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.feature_extraction.text import CountVectorizer

# Load dataset (directly from a common source)
url = "https://raw.githubusercontent.com/Ankit152/IMDB-Sentiment-Analysis/master/IMDB-Dataset.csv"
df = pd.read_csv(url).sample(10000, random_state=42) # Subset for speed

# Encode target
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(f"Dataset loaded with {df.shape[0]} reviews.")
df.head()

Dataset loaded with 10000 reviews.


,review,sentiment,label
33553,I really liked this Summerslam due to the look...,positive,1
9427,Not many television shows appeal to quite as m...,positive,1
199,The film quickly gets to a major chase scene w...,negative,0
12447,Jane Austen would definitely approve of this o...,positive,1
39489,Expectations were somewhat high for me when I ...,negative,0


In [2]:
df['label'].value_counts()

label
1    5039
0    4961
Name: count, dtype: int64

## 2. The "Baseline" Model
To know if our features are actually good, we need a baseline. This uses a simple Bag of Words (BoW) approach.

In [3]:
# Simple vectorization
vectorizer = CountVectorizer(max_features=1000)
X_baseline = vectorizer.fit_transform(df['review'])
y = df['label']

# Split
X_train, X_test, y_train, y_test = train_test_split(X_baseline, y, test_size=0.2, random_state=42)

# Train a simple Logistic Regression
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)

# Evaluate
preds = baseline_model.predict(X_test)
print(f"Baseline F1-Score: {f1_score(y_test, preds):.4f}")

Baseline F1-Score: 0.8542


## 3. The Challenge: Feature Engineering Sandbox
Your goal is to create a function that extracts new numerical features from the raw text. Think about:

Metadata: Length, punctuation, capitalization.

Lexicons: Positive/Negative word counts.

Context: Handling "not" or "never."

In [5]:
def extract_custom_features(text):
    """
    Students: Edit this function to create your features!
    Return a list or numpy array of numbers.
    """
    features = []

    # Example Feature 1: Word Count
    words = text.split()
    features.append(len(words))

    # Example Feature 2: Count of Exclamation Marks
    features.append(text.count('!'))

    # Example Feature 3: 'No/Not' density (Simple Negation)
    negations = len(re.findall(r'\b(not|no|never|neither|nor)\b', text.lower()))
    features.append(negations)

    # --- ADD YOUR OWN IDEAS BELOW ---
    # Feature 4: Average word length
    avg_word_length = np.mean([len(word) for word in words]) if words else 0
    features.append(avg_word_length)

    # Feature 5: Upper case ratio
    upper_count = sum(1 for c in text if c.isupper())
    features.append(upper_count / len(text) if len(text) > 0 else 0)

    # Feature 6: Question marks count
    features.append(text.count('?'))

    # Feature 7: Positive sentiment words (simple list)
    positive_words = ['good', 'great', 'excellent', 'amazing', 'love', 'best', 'fantastic', 'wonderful']
    positive_count = sum(1 for word in words if word.lower() in positive_words)
    features.append(positive_count)

    # Feature 8: Negative sentiment words (simple list)
    negative_words = ['bad', 'awful', 'terrible', 'hate', 'worst', 'horrible', 'poor', 'waste']
    negative_count = sum(1 for word in words if word.lower() in negative_words)
    features.append(negative_count)

    return features

# Apply your features to the dataframe
custom_features_list = df['review'].apply(extract_custom_features).tolist()
X_custom = np.array(custom_features_list)

print(f"New feature matrix shape: {X_custom.shape}")

New feature matrix shape: (10000, 8)


Additional Tips:

- Check Stop words
- TF-IDF

In [9]:

def extract_custom_features2(text):
    features = []
    # Check Stop Words Count
    checkStopWords = ['the', 'is', 'in', 'and', 'to', 'of']
    checkStopWordsCount = sum(1 for word in text.lower().split() if word in checkStopWords)
    features.append(checkStopWordsCount)
    
    # TF-IDF
    tfidf_vectorizer = CountVectorizer(max_features=100)
    tfidf_matrix = tfidf_vectorizer.fit_transform([text])
    features.extend(tfidf_matrix.toarray()[0])
    
    return features